In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')

# --- Get the repo's helper code (utils/funcs.py) into Colab ---
# First run: clone the repo. Later runs: pull, so edits pushed to GitHub show up.
if not os.path.exists('ImageClassification'):
    !git clone https://github.com/durak-dev/ImageClassification.git
else:
    !git -C ImageClassification pull

# Put the repo folder first on the import path, so `utils` means OUR utils folder
sys.path.insert(0, 'ImageClassification')

# Re-read funcs.py automatically after a `git pull` (no runtime restart needed)
%load_ext autoreload
%autoreload 2

from utils.funcs import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model, Sequential
# CNN and classifier layers
from tensorflow.keras.layers import (
    Conv2D,       # Apply convolution filters to an image
    MaxPooling2D, # Reduce the spatial dimensions of feature maps
    Flatten,      # Convert feature maps into a one-dimensional vector
    Dense,        # Connect units to classify the image
    Dropout       # Randomly deactivate units during training
)

# EarlyStopping: stop training when validation performance stops improving
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

Cloning into 'ImageClassification'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


ModuleNotFoundError: No module named 'imp'

In [ ]:
url = 'https://drive.google.com/file/d/1nD6OujVURzL-nEKvsf_KO-nhsQlQSbuw/view?usp=sharing'
df = download_dataset(url)
display(df.head())

In [ ]:
missing = ~df.path.map(os.path.exists)
print(missing.sum(), "missing files")

In [ ]:
df['class_id'] = df['class_id'].map(lambda x: int(x[1:]) if str(x).startswith('n') else x)

In [ ]:
unique_pairs = df[['class_id', 'description']].drop_duplicates()

conflicts_by_id = unique_pairs[unique_pairs.duplicated(subset=['class_id'], keep=False)].sort_values('class_id')

conflicts_by_desc = unique_pairs[unique_pairs.duplicated(subset=['description'], keep=False)].sort_values('description')

print("Conflicting ID mappings:")
print(conflicts_by_id)

Conflicting ID mappings:
Empty DataFrame
Columns: [class_id, description]
Index: []


In [ ]:
plot_images(df, n=64, shuffle=True, seed=420, title_cols='description')

In [ ]:
counts = plot_class_counts(df)

In [ ]:
model, proc, dev = load_clip()

# One text prompt per class, e.g. "a photo of a barn"
classes = df.drop_duplicates('class_id')[['class_id', 'description']].reset_index(drop=True)
prompts = [f"a photo of a {first_name(d)}" for d in classes.description]
txt = clip_text_features(model, proc, dev, prompts)

# Up to 20 images per class, so this runs in reasonable time
samp = df.groupby('class_id', group_keys=False).apply(lambda g: g.sample(min(20, len(g)), random_state=0))
cid2i = {c: i for i, c in enumerate(classes.class_id)}
y_true = samp.class_id.map(cid2i).values

img = clip_image_features(model, proc, dev, samp.path.values)
sims = img @ txt.T                                             # (N images, n classes) cosine similarities
y_pred = sims.argmax(1).numpy()
clip_score = (model.logit_scale.exp().item() * sims).softmax(1).max(1).values.numpy()
print("top 1:", np.mean(y_true == y_pred),
      "top 5:", (sims.topk(5).indices.numpy() == y_true[:, None]).any(1).mean())

In [ ]:
nbr_rank = class_neighbor_ranks(txt)   # nbr_rank[i, j] == 1: j is the class closest in meaning to i
K = 5

In [ ]:
n = len(classes)
cmn = plot_confusion(y_true, y_pred, n)

# Per-class: where does each folder mostly go?
names = classes.description.map(first_name).values
top = cmn.argmax(1)
summary = pd.DataFrame({
    'original': names,
    'clip_top': names[top],
    'frac_top': cmn.max(1),
    'frac_self': np.diag(cmn),
})

In [ ]:
bad = summary[top != np.arange(n)].copy()          # indices, so two classes with the same first word can't collide
bad['nbr_rank'] = nbr_rank[bad.index, top[bad.index]]

suspect_cls = bad[bad.nbr_rank > K].sort_values('frac_top', ascending=False)
neighbor_cls = bad[bad.nbr_rank <= K].sort_values('frac_top', ascending=False)
print(f"{len(suspect_cls)} suspect classes, {len(neighbor_cls)} explained by semantic neighbours")
display(suspect_cls.head(40))

In [ ]:
res = pd.DataFrame({
    'path': samp.path.values,
    'original': names[y_true],
    'clip_pred': names[y_pred],
    'clip_score': clip_score,
    'nbr_rank': nbr_rank[y_true, y_pred],
})
res['kind'] = np.where(y_true == y_pred, 'agree',
               np.where(res.nbr_rank <= K, 'neighbor', 'suspect'))
print(res.kind.value_counts())

review = res[res.kind == 'suspect'].sort_values('clip_score', ascending=False)
display(review.head(30))      # most confident, semantically unrelated disagreements first

In [ ]:
plot_images(review, n=64, title_cols=['original', 'clip_pred'])